# Model Demo Presentation

In [50]:
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown

# 1. Load Data
df = pd.read_csv("../data/aggregated_rolling_features.csv")

# 2. Recreate diff features
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]
for stat in stats:
    df[f"{stat}_diff"] = (
        df[f"previous_10_games_team1_average_{stat}"]
        - df[f"previous_10_games_team2_average_{stat}"]
    )

df["map_score_diff"] = (
    df["team1_previous_10_average_map_score"]
    - df["team2_previous_10_average_map_score"]
)

lasso_features = [
    "kills_diff", "deaths_diff", "assists_diff", "adr_diff", 
    "kast_diff", "kddiff_diff", "map_score_diff"
]
knn_features = [f'{stat}_diff' for stat in stats] + ['map_score_diff', 'bestOf']
knn_features = [c for c in knn_features if c in df.columns]

entry_id = 5078
raw_entry = df.loc[entry_id]

lasso_entry = raw_entry[lasso_features].to_frame().T
knn_entry = raw_entry[knn_features].to_frame().T

print("=== Entry 5078 Attributes ===")
display(lasso_entry.round(3))

=== Entry 5078 Attributes ===


,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff
5078,1.96,0.66,0.06,4.114,0.762,1.3,-1.1


In [51]:
lasso_model = joblib.load("models/best_lasso.pkl")
knn_model = joblib.load("models/original_knn_model.pkl")
print("Models loaded successfully.")

Models loaded successfully.


In [52]:
# KNN Prediction
print("--- K-Nearest Neighbors (KNN) Prediction ---")
knn_pred = knn_model.predict(knn_entry)[0]

if knn_pred == 1:
    result = "Team 1 is predicted to **WIN**."
else:
    result = "Team 1 is predicted to **LOSE**."

display(Markdown(f"**Prediction:** {result}"))

--- K-Nearest Neighbors (KNN) Prediction ---


**Prediction:** Team 1 is predicted to **WIN**.

In [53]:
# Lasso Regression Prediction
print("--- Lasso Regression Prediction ---")
lasso_pred = lasso_model.predict(lasso_entry)[0]

if lasso_pred == 1:
    result = "Team 1 is predicted to **WIN**."
else:
    result = "Team 1 is predicted to **LOSE**."

display(Markdown(f"**Prediction:** {result}"))

--- Lasso Regression Prediction ---


**Prediction:** Team 1 is predicted to **WIN**.

In [54]:
# Actual Result
actual_win = df.loc[entry_id, "team1_win"]

if actual_win == 1:
    actual_result = "Team 1 actually **WON**."
else:
    actual_result = "Team 1 actually **LOST**."

print("--- Actual Match Outcome ---")
display(Markdown(f"{actual_result}"))

--- Actual Match Outcome ---


Team 1 actually **LOST**.